In [0]:
%run ./utils

# Imports

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
# Converte texto de horário no padrão dd/MM/yyyy às HH:mm:ss em timestamp
def parse_datetime(column):
    return F.to_timestamp(
        F.concat(
            F.regexp_extract(column, r"(\d{2}/\d{2}/\d{4})", 1),
            F.lit(" "),
            F.regexp_extract(column, r"(\d{2}:\d{2}:\d{2})", 1)
        ),
        "dd/MM/yyyy HH:mm:ss"
    )

# Load

In [0]:
dbutils.widgets.text("bronze_table_name", "tb_chamados_hora")
bronze_table_name = dbutils.widgets.get("bronze_table_name")

In [0]:
df_chamados_hora = (
    spark.table(f"{bronze_schema}.{bronze_table_name}")
        .select(
            "ID_Chamado",
            "ID_Cliente",
            "Hora_Abertura_Chamado",
            "Hora_Inicio_Atendimento",
            "Hora_Finalizacao_Atendimento",
            "ingestion_datetime"
        ).distinct()
        .orderBy("ID_Chamado")
)

In [0]:
df_chamados_hora.count()

# Deduplicação

In [0]:
# Janela agrupada pela chave primária e ordenada pelo timestamp de ingestão mais recente
window_spec = Window.partitionBy("ID_Chamado").orderBy(F.col("ingestion_datetime").desc())

In [0]:
df_dedup = (
    df_chamados_hora
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Tratamento

In [0]:
# Transforma id em string
df_cast = cast_id_to_string(df_dedup)

In [0]:
# Converte os textos de horário em timestamp usando o padrão textual da base
df_horas = (
    df_cast
    .withColumn("hora_abertura_chamado", parse_datetime(F.col("Hora_Abertura_Chamado")))
    .withColumn("hora_inicio_atendimento", parse_datetime(F.col("Hora_Inicio_Atendimento")))
    .withColumn("hora_finalizacao_atendimento", parse_datetime(F.col("Hora_Finalizacao_Atendimento")))
)

In [0]:
# Corrige o ano da abertura quando diverge do ano de referência mantendo o mesmo dia e mês
referencia = F.coalesce(F.col("hora_inicio_atendimento"), F.col("hora_finalizacao_atendimento"))
df_ano_corrigido = df_horas.withColumn(
    "hora_abertura_chamado",
    F.when(
        (F.year("hora_abertura_chamado") != F.year(referencia))
        & (F.month("hora_abertura_chamado") == F.month(referencia))
        & (F.dayofmonth("hora_abertura_chamado") == F.dayofmonth(referencia)),
        F.make_timestamp(
            F.year(referencia),
            F.month("hora_abertura_chamado"),
            F.dayofmonth("hora_abertura_chamado"),
            F.hour("hora_abertura_chamado"),
            F.minute("hora_abertura_chamado"),
            F.second("hora_abertura_chamado")
        )
    ).otherwise(F.col("hora_abertura_chamado"))
)

In [0]:
# Acrescenta um dia quando a finalização foi registrada antes do início por virada de meia-noite
df_fim_corrigido = df_ano_corrigido.withColumn(
    "hora_finalizacao_atendimento",
    F.when(
        F.col("hora_finalizacao_atendimento") < F.col("hora_inicio_atendimento"),
        F.col("hora_finalizacao_atendimento") + F.expr("INTERVAL 1 DAY")
    ).otherwise(F.col("hora_finalizacao_atendimento"))
)

In [0]:
# Calcula espera e atendimento em segundos a partir dos horários já tratados
df_tempos = df_fim_corrigido.withColumn(
    "tempo_espera_segundos",
    F.unix_timestamp("hora_inicio_atendimento") - F.unix_timestamp("hora_abertura_chamado")
).withColumn(
    "tempo_atendimento_segundos",
    F.unix_timestamp("hora_finalizacao_atendimento") - F.unix_timestamp("hora_inicio_atendimento")
)

In [0]:
# Renomeia as colunas para o padrão do catálogo e define a ordem final
df_limpo = df_tempos.select(
    F.col("ID_Chamado").alias("id_chamado"),
    F.col("ID_Cliente").alias("id_cliente"),
    "hora_abertura_chamado",
    "hora_inicio_atendimento",
    "hora_finalizacao_atendimento",
    "tempo_espera_segundos",
    "tempo_atendimento_segundos",
    "ingestion_datetime"
).orderBy("id_chamado")

# Salvamento

In [0]:
dbutils.widgets.text("silver_table_name", "tb_chamados_hora")
silver_table_name = dbutils.widgets.get("silver_table_name")

In [0]:
save_df(df_limpo, silver_table_name)